# B10–B14耗电规律与Bideal校准草案

## tl;dr
当前配对下，B13呈现高区间快、medium慢、low再次变快的特征。五个电池low40→20均比medium75→40掉电更快，但加速比例不同。
B12在9月6日仅采用D5数据，D2那次已按用户要求排除。当前仍是电池–机体配对的SOC分段Bideal草案，跨机体通用性尚未验证。


## Context & Methods
### Key Assumptions
用户确认2026-09-06单独Hover baseline均无风且设置相同高度；lv1是默认标签。
以最新同日、当前常用电池–机体配对为主要校准队列：B11/D1、B10/D2、B13/D3、B14/D4、B12/D5。
按用户要求排除20260906_165812（B12/D2），不用于本分析的统计、曲线或对照；原始文件保留，排除规则固定在分析脚本中。
高区间先用95%→75%以避开满电平台，并保证跨机可比较；不能声称覆盖100%→95%。Medium为75%→40%，Low为40%→20%。
只使用真实hover_to_10_percent记录，起降不计入。平均率=SOC下降百分点×60/首次观察到两个边界的时间差。
不插值、不拼接、不补数据。边界跳过、时间断点>5秒、单次SOC跳变>2个百分点等排除。
单位是设备报告的SOC百分点/分钟，不是Wh/min；每条独立飞行是一个样本，不把每秒记录当独立重复。
OLS斜率作为敏感性检查。分区均率不意味着区间内部严格线性。
历史记录分开列出；Wind Tunnel只作辅助检查，不用于分离电池与位置/机体效应。


## Data
只读取原始数据库与注册信息，输出另存，不修改原始实验。

In [1]:
import sys
from pathlib import Path
root=Path('/Users/alexmason/Downloads/drone_experiment')
sys.path.insert(0,str(root/'output_py'))
from analyze_bideal_20260908 import analyze
result=analyze()
print(result['inventory'].status.value_counts().to_string())
print('原始Wind Tunnel文件数:',len(result['wind_inventory']))


status
observed_hover      21
no_hover            10
excluded_by_user     1
原始Wind Tunnel文件数: 163


## Results
### 最新同日Hover：按当前配对选取，每配对一个样本

In [2]:
rates=result['coefficients'].pivot(index='battery_id',columns='band',values='rate_pp_min')
print(rates.round(3).to_string())
print('low/medium倍率:')
print((rates.low40_20/rates.medium75_40).round(3).to_string())


band        high95_75  low40_20  medium75_40
battery_id                                  
B10             6.986    12.192        9.332
B11             7.197    13.553        9.377
B12            10.667    15.479       10.446
B13            21.324    14.530        7.358
B14             8.785    14.395        8.859
low/medium倍率:
battery_id
B10    1.307
B11    1.445
B12    1.482
B13    1.975
B14    1.625


### B12采用D5数据及其余历史记录（已排除9月6日D2那次）

In [3]:
r=result['recent']
print(r[r.battery_id.eq('B12')][['drone','band','rate_pp_min','tof_median']].round(3).to_string(index=False))
b=result['bands']
print(b[b.valid & b.band.ne('low20_10')].pivot_table(index=['battery_id','drone','run_id'],columns='band',values='rate_pp_min').round(3).to_string())


  drone        band  rate_pp_min  tof_median
drone_5   high95_75       10.667        82.0
drone_5 medium75_40       10.446        82.0
drone_5    low40_20       15.479        82.0
drone_5    low20_10        7.087        82.0
band                                               high95_75  low40_20  medium75_40
battery_id drone   run_id                                                          
B10        drone_2 20260513_180556                     9.954    11.560        8.155
                   20260906_173013                     6.986    12.192        9.332
           drone_3 batterytest_10_20260506_115206      9.265    10.809        8.642
B11        drone_1 20260513_143201                    21.722    14.168        7.798
                   20260906_164725                     7.197    13.553        9.377
B12        drone_2 20260513_145311                     8.069    11.478        9.128
                   20260513_154650                    10.209    12.434        8.280
                   

### 可追溯草案与独立计算复核

In [4]:
import numpy as np
c=result['coefficients']
assert np.allclose(c.rate_pp_min*c.physical_to_bideal_scale,c.bideal_pp_min)
for _,row in c.iterrows():
    raw=pd.read_csv(root/row.source) if 'pd' in globals() else __import__('pandas').read_csv(root/row.source)
    raw=raw[raw.phase.eq('hover_to_10_percent')]
    start=raw.loc[raw.battery.le(row.upper),'elapsed_time'].iloc[0]
    end=raw.loc[(raw.elapsed_time>start)&raw.battery.le(row.lower),'elapsed_time'].iloc[0]
    assert abs((row.upper-row.lower)*60/(end-start)-row.rate_pp_min)<1e-10
print('15个原始区间阈值复核通过；归一化恒等式通过。')
print(c[['battery_id','band','bideal_pp_min','physical_to_bideal_scale']].round(4).to_string(index=False))
print('Wind Tunnel同次飞行low/medium辅助对照（有删失，不作为电池校准）:')
print(result['wind_pairs'].groupby('battery_id').low_medium_multiplier.agg(['count','median']).round(3).to_string())


15个原始区间阈值复核通过；归一化恒等式通过。
battery_id        band  bideal_pp_min  physical_to_bideal_scale
       B11   high95_75        10.9915                    1.5273
       B11 medium75_40         9.0742                    0.9677
       B11    low40_20        14.0300                    1.0352
       B10   high95_75        10.9915                    1.5735
       B10 medium75_40         9.0742                    0.9724
       B10    low40_20        14.0300                    1.1507
       B13   high95_75        10.9915                    0.5155
       B13 medium75_40         9.0742                    1.2332
       B13    low40_20        14.0300                    0.9656
       B14   high95_75        10.9915                    1.2512
       B14 medium75_40         9.0742                    1.0243
       B14    low40_20        14.0300                    0.9747
       B12   high95_75        10.9915                    1.0305
       B12 medium75_40         9.0742                    0.8687
       B12    lo

## Takeaways
第一版只发布可追溯的草案，不覆盖既有冻结表和飞行控制/训练代码。以同日五个常用配对分区率的等权均值定义Bideal参考率，a_b,k=r_ideal,k/r_b,k。
同一SOC区间内归一化下降量：DeltaSOC_ideal=a_b,k*DeltaSOC_reported。跨区间需拆段求和；这不是直接把绝对SOC乘系数。
迁移到不同机体、风力和负载是待验证假设。高区间历史变化大；每个当前配对同日仅一次完整样本，暂不作为最终通用系数。


## 三段连续直线模型
## 已生成三条连续直线组成的Bideal
High：100→75%，Medium：75→40%，Low：40→0%；分别使用固定的下降速度。曲线在75%和40%处相接，没有跳变。
这是一条人为定义的参考电池模型，不是新增实测数据。High斜率用95→75%的数据计算并延至100%；Low斜率用40→20%的数据计算，20→0仅为未验证延长线。默认计算拒绝进入20%以下。
|阶段|下降速度（百分点/分钟）|模型累计时间（秒）|
|---|---:|---:|
|high|10.9915|0.00–136.47|
|medium|9.0742|136.47–367.89|
|low|14.0300|367.89–538.96|


In [5]:
from output_py.build_bideal_three_stage import build_model, validate, soc_after
model, rates, diagnostics = build_model()
print(validate(model,rates))
print(__import__('pandas').DataFrame(model['segments'])[['stage','rate_pp_min','start_s','end_s']].round(4).to_string(index=False))


{'status': 'passed', 'raw_threshold_checks': 15, 'continuous_boundaries': 2, 'monotonicity_grid_points': 501, 'below20_guard': True, 'normalization_additivity': True}
 stage  rate_pp_min  start_s    end_s
  high      10.9915   0.0000 136.4688
medium       9.0742 136.4688 367.8945
   low      14.0300 367.8945 538.9567
